In [1]:
import numpy as np
import pandas as pd

# Import modules for splitting data, encoding categorical features,
# handling missing values, and model training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('/content/train.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Drop columns that are not relevant for the prediction task from the DataFrame
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace = True)

In [5]:
# Split the DataFrame into training and testing sets
# X contains features (all columns except 'Survived'), y contains the target ('Survived')
# test_size=0.2 means 20% of the data will be used for testing, random_state ensures reproducibility
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [6]:
# Check for the number of missing values in each column of the DataFrame
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [7]:
# Initialize SimpleImputer for 'Age' (using mean strategy for numerical data)
si_age = SimpleImputer()
# Initialize SimpleImputer for 'Embarked' (using most frequent strategy for categorical data)
si_embarked = SimpleImputer(strategy='most_frequent')

# Apply imputation to the 'Age' column in the training set
X_train_age = si_age.fit_transform(X_train[['Age']])
# Apply imputation to the 'Embarked' column in the training set
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

# Apply imputation to the 'Age' column in the test set
X_test_age = si_age.fit_transform(X_test[['Age']])
# Apply imputation to the 'Embarked' column in the test set
X_test_embarked = si_embarked.fit_transform(X_test[['Embarked']])

In [8]:
# Initialize OneHotEncoder for 'Sex' (sparse_output=False returns a dense array)
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
# Initialize OneHotEncoder for 'Embarked'
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

# Apply one-hot encoding to 'Sex' in the training set
X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
# Apply one-hot encoding to the imputed 'Embarked' in the training set
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

# Apply one-hot encoding to 'Sex' in the test set
X_test_sex = ohe_sex.fit_transform(X_test[['Sex']])
# Apply one-hot encoding to the imputed 'Embarked' in the test set
X_test_embarked = ohe_embarked.fit_transform(X_test_embarked)

In [9]:
# Create a copy of the training features, dropping the original 'Sex', 'Age', and 'Embarked' columns
# These columns have been transformed and will be re-added later
X_train_rem = X_train.drop(columns=['Sex', 'Age', 'Embarked'])

In [10]:
# Create a copy of the test features, dropping the original 'Sex', 'Age', and 'Embarked' columns
# These columns have been transformed and will be re-added later
X_test_rem = X_test.drop(columns=['Sex', 'Age', 'Embarked'])

In [11]:
# Concatenate the remaining training features with the imputed 'Age' and one-hot encoded 'Sex' and 'Embarked'
X_train_transformed = np.concatenate((X_train_rem,X_train_age,X_train_sex,X_train_embarked),axis=1)
# Concatenate the remaining test features with the imputed 'Age' and one-hot encoded 'Sex' and 'Embarked'
X_test_transformed = np.concatenate((X_test_rem,X_test_age,X_test_sex,X_test_embarked),axis=1)

In [12]:
# Initialize a Decision Tree Classifier model
clf = DecisionTreeClassifier()
# Train the classifier using the transformed training features and target labels
clf.fit(X_train_transformed,y_train)

DecisionTreeClassifier()

In [13]:
# Use the trained classifier to make predictions on the transformed test set
y_pred = clf.predict(X_test_transformed)

In [14]:
# Display the array of predicted values for the test set
y_pred

array([0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

In [15]:
# Import the accuracy_score metric to evaluate model performance
from sklearn.metrics import accuracy_score
# Calculate and display the accuracy score by comparing actual test labels (y_test) with predicted labels (y_pred)
accuracy_score(y_test,y_pred)

0.7597765363128491

In [16]:
# Import the pickle module for serializing and deserializing Python objects
import pickle

In [17]:
# Save the trained OneHotEncoder for 'Sex' to a file named 'ohe_sex.pkl'
pickle.dump(ohe_sex,open('ohe_sex.pkl','wb'))
# Save the trained OneHotEncoder for 'Embarked' to a file named 'ohe_embarked.pkl'
pickle.dump(ohe_embarked,open('ohe_embarked.pkl','wb'))
# Save the trained DecisionTreeClassifier model to a file named 'clf.pkl'
pickle.dump(clf,open('clf.pkl','wb'))